In [26]:
from datascience import *
import matplotlib
path_data = '../../../assets/data/'
matplotlib.use('Agg')
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import numpy as np

Population = Table.read_table(path_data + 'Canada-Population.csv')

# Manipulation and Transformation of Tabular data

In data science, we often need to perform complex transformations and calculations on our data. So far we have seen some examples of creating new columns of tables by applying functions to existing columns or to other arrays. All of those functions took arrays as their arguments. But frequently we will want to convert the entries in a column by a function that doesn't take an array as its argument. 


For example, it might take just one number as its argument, as in the function `categorize_score` defined below.

In [27]:
#the function to categorize scores
def categorize_score(score):
    if score >= 90:
        return 'A'
    elif score >= 80:
        return 'B'
    elif score >= 70:
        return 'C'
    elif score >= 60:
        return 'D'
    else:
        return 'F'

In [28]:
categorize_score(17)

'F'

In [29]:
categorize_score(97)

'A'

In [30]:
categorize_score(76)

'C'

The function `categorize_score` simply returns letter grade given a score. Let's consider a Table containing information about students and their scores in a particular exam. To use this function on many scores at once, we will have to be able to *refer* to the function itself, without actually calling it. Analogously, we might show a cake recipe to a chef and ask her to use it to bake 6 cakes.  In that scenario, we are not using the recipe to bake any cakes ourselves; our role is merely to refer the chef to the recipe.  Similarly, we can ask a table to call `categorize_score` on 6 different numbers in a column.

First, we create the table `students` with a column for Name and Score. 

In [31]:
students = Table().with_columns(
    'Name', make_array('Alice', 'Bob', 'Charlie', 'David'),
    'Score', make_array(85, 92, 78, 64)
)
students

Name,Score
Alice,85
Bob,92
Charlie,78
David,64


## Applying a Function to a Column

To convert each of the scores to it's letter grade, we will use a new Table method. The `apply` method calls a function on each element of a column, forming a new array of return values. To indicate which function to call, just name it (without quotation marks or parentheses). The name of the column of input values is a string that must still appear within quotation marks.

In [32]:
students.apply(categorize_score, 'Score')

array(['B', 'A', 'C', 'D'],
      dtype='<U1')

What we have done here is `apply` the function `categorize_score` to each value in the `Score` column of the table `students`. The output is the array of corresponding return values of the function. For example, 85 became 'B', 92 became 'A' and so on.

This array, which has the same length as the original `Score` column of the `students` table, can be used as the values in a new column called `Letter Grad` alongside the existing `Name` and `Score` columns.

In [33]:
students.with_column(
    'Letter Grade', students.apply(categorize_score, 'Score')
)

Name,Score,Letter Grade
Alice,85,B
Bob,92,A
Charlie,78,C
David,64,D


## Functions as Values
We've seen that Python has many kinds of values.  For example, `6` is a number value, `"cake"` is a text value, `Table()` is an empty table, and `ages` is a name for a table value (since we defined it above).

In Python, every function, including `categorize_score`, is also a value. It helps to think about recipes again. A recipe for cake is a real thing, distinct from cakes or ingredients, and you can give it a name like "Ani's cake recipe." When we defined `categorize_score` with a `def` statement, we actually did two separate things: we created a function that converts a score to letter grade, and we gave it the name `categorize_score`.

We can refer to any function by writing its name, without the parentheses or arguments necessary to actually call it. We did this when we called `apply` above.  When we write a function's name by itself as the last line in a cell, Python produces a text representation of the function, just like it would print out a number or a string value.

In [34]:
categorize_score

<function __main__.categorize_score(score)>

Notice that we did not write `"categorize_score"` with quotes (which is just a piece of text), or `categorize_score()` (which is a function call, and an invalid one at that).  We simply wrote `categorize_score` to refer to the function.

Just like we can define new names for other values, we can define new names for functions.  For example, suppose we want to refer to our function as `categorize` instead of `categorize_score`.  We can just write this:

In [35]:
categorize = categorize_score

Now `categorize` is a name for a function.  It's the same function as `categorize_score`, so the printed value is exactly the same.

In [36]:
categorize

<function __main__.categorize_score(score)>

Let us see another application of `apply`.

## Applying Functions to Rows

Sometimes, we may need to apply a function to a row instead of a column. Let's see an example. For this example, we will redefine our initial `students` table with an additional column.

In [37]:
students = Table().with_columns(
    'Name', make_array('Alice', 'Bob', 'Charlie', 'David'),
    'Score', make_array(85, 92, 78, 64),
    'Quiz', make_array(95, 70, 91, 72)
)
students

Name,Score,Quiz
Alice,85,95
Bob,92,70
Charlie,78,91
David,64,72


- Let's say that, the 'Quiz' is worth 10% of their final grade.
- Let's create a column called **Final_Grade**. To do this, we need to compute:

> (0.9) * (Score) + (0.1) * (Quiz)

- This formula relies on two columns!
- Note that when applying to a row, we call apply() on the whole table, not just a single column!

In [38]:
def computeFinalGrade(row):
	Score = row[1]
	Quiz = row[2]

	final_grade = (0.9*Score) + (0.1*Quiz)
	return final_grade

students.with_column('Final_Grade',  students.apply(computeFinalGrade))

Name,Score,Quiz,Final_Grade
Alice,85,95,86
Bob,92,70,89.8
Charlie,78,91,79.3
David,64,72,64.8


## Example: Canada Population

One of the most prevalent applications of the `apply` function is in data preprocessing. While we will explore data preprocessing techniques in greater detail in subsequent chapters, this section provides a brief example to illustrate how `apply` can be utilized for preprocessing tasks.

We will take Canada's population dataset (in the years 2021 to 2022) for preprocessing.

In [39]:
Population

Geography,Jan 2021,Jan 2022
Canada,"38,058,291","38,567,576"
Newfoundland and Labrador,"525,895","528,977"
Prince Edward Island,"159,240","164,195"
Nova Scotia,"990,025","1,010,460"
New Brunswick,"784,950","798,656"
Quebec,"8,550,561","8,613,999"
Ontario,"14,772,726","14,999,441"
Manitoba,"1,383,854","1,400,663"
Saskatchewan,"1,166,348","1,171,031"
Alberta,"4,418,338","4,465,537"


If we want to calculate the population growth fron year 2021 to 2022 it will raise a Typeerror. 

**Note: to perform arithmetic operation between two columns we can also use `apply`.**

In [40]:
Population.apply(lambda x, y: x - y, 'Jan 2022', 'Jan 2021')

TypeError: unsupported operand type(s) for -: 'numpy.str_' and 'numpy.str_'

Let's examine the type of each column to understand why the arithmetic operation is causing an error. This issue arises because `datascience` has interpreted the columns as strings instead of numbers, and arithmetic operations cannot be performed on string data types.

As you can see the values of these two columns contain commas, these values are interpreted as strings by `datascience`. To perform arithmetic operations, we need to remove the commas to convert these columns to numeric data types. Here we will use `apply` to convert string to numeric data.

In [41]:
def to_number(str_count):
    #replace is a method of string type in python
    num_count = int(str_count.replace(',',''))
    return num_count


Population = Population.with_column('Jan 2022', Population.apply(to_number, 'Jan 2022'))
Population = Population.with_column('Jan 2021', Population.apply(to_number, 'Jan 2021'))
Population

Geography,Jan 2021,Jan 2022
Canada,38058291,38567576
Newfoundland and Labrador,525895,528977
Prince Edward Island,159240,164195
Nova Scotia,990025,1010460
New Brunswick,784950,798656
Quebec,8550561,8613999
Ontario,14772726,14999441
Manitoba,1383854,1400663
Saskatchewan,1166348,1171031
Alberta,4418338,4465537


Now as  both columns 'Jan 2022' and 'Jan 2021' are integer type we can calculate the percentage of population growth within one year. 

In [42]:
Population = Population.with_column('%Growth',Population.apply(lambda x, y: (x - y)/y*100, 'Jan 2022', 'Jan 2021'))
Population

Geography,Jan 2021,Jan 2022,%Growth
Canada,38058291,38567576,1.33817
Newfoundland and Labrador,525895,528977,0.586049
Prince Edward Island,159240,164195,3.11166
Nova Scotia,990025,1010460,2.06409
New Brunswick,784950,798656,1.7461
Quebec,8550561,8613999,0.741916
Ontario,14772726,14999441,1.53469
Manitoba,1383854,1400663,1.21465
Saskatchewan,1166348,1171031,0.40151
Alberta,4418338,4465537,1.06825


In this section, we introduced the concept of manipulating and transforming tables using the `apply` method. We demonstrated that `apply` can be utilized both column-wise and row-wise. Finally, we presented an example to illustrate the usability of `apply` on tabular data.